# 00 — Environment Setup & Project Overview

This notebook sets up the environment for the Transformer Math Lab and outlines the structure of the entire project.

### What we'll cover
- Install and import required Python libraries
- Set random seeds and configure plotting utilities
- Project roadmap: attention → MHA → positional encodings → FFN → normalization → full transformer
- How the notebooks are organized and how to navigate them

This file acts as the starting point before diving into the math of transformers.


## Imports

This cell loads all core libraries used throughout the entire transformer analysis project.

- **math, random, numpy** — numerical operations and randomness
- **torch, torch.nn, torch.nn.functional** — tensor math and neural-network building blocks
- **matplotlib** — for visualizing matrices, attention maps, and embeddings

These imports are intentionally lightweight and stable across all notebooks to keep the environment consistent.

In [3]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["font.size"] = 12

print("Imports loaded.")

Imports loaded.


## ⚙️ Device Selection (M1 Mac / CPU)

Transformers rely heavily on matrix operations, so it's important to choose the fastest available compute device.

This cell:

- Detects whether the **MPS backend** (Apple Metal Performance Shaders) is available.  
  - MPS is the GPU backend for M1/M2 Macs.
- Falls back to **CPU** if MPS is not available.
- Stores the selected device in the `device` variable, which future notebooks can use when creating tensors or models.

All subsequent notebooks (01–10) can reliably use this `device` variable to ensure consistent behavior across runs and machines.

In [7]:
# Device selection: Use MPS on Apple Silicon if available, otherwise CPU

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("✔️ Running on Apple Silicon MPS backend")
else:
    device = torch.device("cpu")
    print("⚠️ Running on CPU")

device

✔️ Running on Apple Silicon MPS backend


device(type='mps')

## 🎲 Reproducibility: Random Seed Setup

To ensure consistent results across runs and across notebooks, we set a global random seed.

This cell:

- Sets the seed for Python’s built-in `random` module  
- Sets the seed for NumPy  
- Sets the seed for PyTorch  
- If available, also seeds the **MPS backend** on Apple Silicon

Why this matters:

- The same initial tensors (Q, K, V) will be generated every time  
- Visualizations (attention maps, positional encodings) will match across runs  
- Experimental results in later notebooks will be consistent and debuggable  

This is essential for mathematical analysis and clear explanations.

In [5]:
# Set a global random seed for reproducibility

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Extra: Seed the MPS (Metal) backend if available
if torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

print("Random seed set to:", seed)

Random seed set to: 42


## 🔍 Visualization Utilities

Throughout this project, we will frequently inspect matrices:

- Attention score matrices  
- Softmax probability distributions  
- Positional encodings  
- Embedding projections  

To make this easier, this cell defines two helper functions:

### `show_matrix(matrix, title)`
Displays any 2D numerical array (NumPy or PyTorch) as a heatmap using a clean color scale.

### `show_attention(attn, title)`
Specifically for visualizing **attention weight matrices**, with axis labels that map queries to keys.

These visualization tools will be used extensively in:
- Notebook 01 (Scaled Dot-Product Attention)
- Notebook 02 (Multi-Head Attention)
- Notebook 03 (Positional Encodings)
- Notebook 07 (Interpreting Attention Maps)

They make the math **visible**, which is crucial for understanding how transformers work internally.

In [6]:
# Utility functions for visualizing matrices and attention maps

def show_matrix(matrix, title="Matrix"):
    """Display a matrix with a colorbar."""
    plt.imshow(matrix, cmap="viridis")
    plt.title(title)
    plt.colorbar()
    plt.show()


def show_attention(attn, title="Attention Weights"):
    """Visualize attention weights."""
    plt.imshow(attn, cmap="inferno")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Key positions")
    plt.ylabel("Query positions")
    plt.show()


print("Utility visualization functions loaded.")

Utility visualization functions loaded.


# 🧭 Project Roadmap

This project explores transformers from the ground up, focusing on the mathematics, mechanics, and intuition behind each component.

The notebooks proceed in the following order:

1. **Scaled Dot-Product Attention**  
2. **Multi-Head Attention**  
3. **Positional Encodings**  
4. **Feed-Forward Networks**  
5. **Layer Normalization & Residual Connections**  
6. **Minimal Transformer Encoder**  
7. **Interpreting Attention Maps**  
8. **Embedding Geometry**  
9. **Training a Tiny Transformer**  
10. **Modern Transformer Variants (RoPE, GPT-style, efficient attention)**  

Notebook **00** sets up a consistent environment so all other notebooks (01–10) behave predictably.

## ♻️ Sharing Code Across All Notebooks

To keep this project clean, consistent, and easy to maintain, all shared code has been moved into a single Python module:

This module contains:

- All common imports (NumPy, PyTorch, Matplotlib)
- Device selection logic (MPS for Apple Silicon or CPU fallback)
- A reproducible seed-setting function (`set_seed`)
- Visualization helpers (`show_matrix`, `show_attention`)

### Why this matters
Instead of repeating the same setup code in every notebook (01–10), each notebook can now load everything with a single import:

```python
from scripts.attention_from_scratch_utils import *

set_seed(42)
print("Using device:", device)